In [ ]:
"""
Parity tests: GS, SK, and NWQEC C++ transpilers must share the same pipeline
shape. Only their RZ-synthesis step is allowed to differ. These tests examine
the circuit at each pipeline stage and assert structural parity.
"""

In [ ]:
from __future__ import annotations

In [ ]:
import math

In [ ]:
import numpy as np
import pytest
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, process_fidelity

In [ ]:
from ftcircuitbench.transpilers._basis import (
    INTERMEDIATE_RZ_BASIS,
    PBC_COMPATIBLE_CLIFFORD_T_BASIS,
    is_clifford_t_basis,
    to_intermediate_rz,
)
from ftcircuitbench.transpilers.gs_transpiler import (
    GRIDSYNTH_INTERMEDIATE_BASIS,
    transpile_to_gridsynth_clifford_t,
)
from ftcircuitbench.transpilers.nwqec_ct import (
    is_nwqec_available,
    transpile_to_clifford_t_cpp,
)
from ftcircuitbench.transpilers.sk_transpiler import (
    INTERMEDIATE_RZ_BASIS as SK_INTERMEDIATE,
)
from ftcircuitbench.transpilers.sk_transpiler import (
    transpile_to_solovay_kitaev_clifford_t,
)

---------------------------------------------------------------------------
Test inputs (kept small + RZ-bearing to exercise the synthesis step)
---------------------------------------------------------------------------

In [ ]:
def _rz_circuit_simple() -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.rz(0.3, 0)
    qc.cx(0, 1)
    return qc

In [ ]:
def _rz_circuit_multi() -> QuantumCircuit:
    qc = QuantumCircuit(3)
    qc.h(0)
    qc.rz(math.pi / 7, 0)
    qc.cx(0, 1)
    qc.rz(math.pi / 9, 1)
    qc.h(2)
    qc.cx(1, 2)
    qc.rz(math.pi / 11, 2)
    return qc

In [ ]:
def _already_clifford_t_circuit() -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.t(0)
    qc.cx(0, 1)
    qc.tdg(1)
    qc.s(0)
    return qc

In [ ]:
_INPUT_FACTORIES = [_rz_circuit_simple, _rz_circuit_multi]

---------------------------------------------------------------------------
Stage 0: shared basis constants are exactly the canonical ones
---------------------------------------------------------------------------

In [ ]:
def test_canonical_intermediate_basis_set() -> None:
    assert set(INTERMEDIATE_RZ_BASIS) == {
        "cx",
        "h",
        "s",
        "sdg",
        "t",
        "tdg",
        "x",
        "y",
        "z",
        "rz",
    }

In [ ]:
def test_canonical_pbc_basis_set() -> None:
    assert set(PBC_COMPATIBLE_CLIFFORD_T_BASIS) == {
        "cx",
        "h",
        "s",
        "sdg",
        "t",
        "tdg",
        "x",
        "y",
        "z",
    }

In [ ]:
def test_each_transpiler_module_uses_canonical_intermediate() -> None:
    """All three modules must agree on the intermediate basis (as a set)."""
    assert set(GRIDSYNTH_INTERMEDIATE_BASIS) == set(INTERMEDIATE_RZ_BASIS)
    assert set(SK_INTERMEDIATE) == set(INTERMEDIATE_RZ_BASIS)
    # nwqec_ct re-exports INTERMEDIATE_RZ_BASIS directly; covered by the
    # canonical constant test above.

---------------------------------------------------------------------------
Stage 1+2: prepare_input + early-return parity
---------------------------------------------------------------------------

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_intermediate_is_strictly_in_canonical_basis_gs(factory) -> None:
    inter, _ = transpile_to_gridsynth_clifford_t(
        factory(), gridsynth_precision=3, return_intermediate=True
    )
    ops = set(inter.count_ops()) - {"barrier"}
    assert ops.issubset(set(INTERMEDIATE_RZ_BASIS)), ops

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_intermediate_is_strictly_in_canonical_basis_sk(factory) -> None:
    inter, _ = transpile_to_solovay_kitaev_clifford_t(
        factory(), recursion_degree=1, return_intermediate=True
    )
    ops = set(inter.count_ops()) - {"barrier"}
    assert ops.issubset(set(INTERMEDIATE_RZ_BASIS)), ops

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_intermediate_is_strictly_in_canonical_basis_nwqec(factory) -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    inter, _ = transpile_to_clifford_t_cpp(
        factory(), epsilon=1e-3, return_intermediate=True, forbid_python_fallback=True
    )
    ops = set(inter.count_ops()) - {"barrier"}
    assert ops.issubset(set(INTERMEDIATE_RZ_BASIS)), ops

---------------------------------------------------------------------------
Stage 3: intermediates are STRUCTURALLY IDENTICAL across pipelines
---------------------------------------------------------------------------

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_intermediates_match_across_pipelines(factory) -> None:
    """Same input + same canonical intermediate basis + opt-level=0 ==> same intermediate."""
    inter_gs, _ = transpile_to_gridsynth_clifford_t(
        factory(), gridsynth_precision=3, return_intermediate=True
    )
    inter_sk, _ = transpile_to_solovay_kitaev_clifford_t(
        factory(), recursion_degree=1, return_intermediate=True
    )

    # Compare gate counts (the canonical observable; metadata may differ).
    assert dict(inter_gs.count_ops()) == dict(inter_sk.count_ops())

    if is_nwqec_available():
        inter_nw, _ = transpile_to_clifford_t_cpp(
            factory(),
            epsilon=1e-3,
            return_intermediate=True,
            forbid_python_fallback=True,
        )
        assert dict(inter_gs.count_ops()) == dict(inter_nw.count_ops())

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_intermediate_preserves_unitary(factory) -> None:
    """Stage-3 transpile to canonical intermediate must be exact (unitary-preserving)."""
    qc = factory()
    inter = to_intermediate_rz(qc.copy())
    fid = process_fidelity(Operator(qc), Operator(inter))
    assert fid == pytest.approx(1.0, abs=1e-9)

---------------------------------------------------------------------------
Stage 4+5: final outputs all land strictly in PBC basis
---------------------------------------------------------------------------

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_final_in_pbc_basis_gs(factory) -> None:
    out = transpile_to_gridsynth_clifford_t(factory(), gridsynth_precision=3)
    assert is_clifford_t_basis(out)

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_final_in_pbc_basis_sk(factory) -> None:
    out = transpile_to_solovay_kitaev_clifford_t(factory(), recursion_degree=1)
    assert is_clifford_t_basis(out)

In [ ]:
@pytest.mark.parametrize("factory", _INPUT_FACTORIES)
def test_final_in_pbc_basis_nwqec(factory) -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    out = transpile_to_clifford_t_cpp(
        factory(), epsilon=1e-3, forbid_python_fallback=True
    )
    assert is_clifford_t_basis(out)

---------------------------------------------------------------------------
Stage 2: early-return parity when input is already in PBC basis
---------------------------------------------------------------------------

In [ ]:
def _ops_dict(qc: QuantumCircuit) -> dict:
    return {k: v for k, v in qc.count_ops().items() if k != "barrier"}

In [ ]:
def test_early_return_gs_passes_input_through_unchanged() -> None:
    qc = _already_clifford_t_circuit()
    inter, out = transpile_to_gridsynth_clifford_t(
        qc.copy(), gridsynth_precision=3, return_intermediate=True
    )
    assert _ops_dict(out) == _ops_dict(qc)
    # Per parity contract, intermediate == input on the early-return path.
    assert _ops_dict(inter) == _ops_dict(qc)

In [ ]:
def test_early_return_sk_passes_input_through_unchanged() -> None:
    qc = _already_clifford_t_circuit()
    inter, out = transpile_to_solovay_kitaev_clifford_t(
        qc.copy(), recursion_degree=1, return_intermediate=True
    )
    assert _ops_dict(out) == _ops_dict(qc)
    assert _ops_dict(inter) == _ops_dict(qc)

In [ ]:
def test_early_return_nwqec_passes_input_through_unchanged() -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    qc = _already_clifford_t_circuit()
    inter, out = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, return_intermediate=True, forbid_python_fallback=True
    )
    assert _ops_dict(out) == _ops_dict(qc)
    assert _ops_dict(inter) == _ops_dict(qc)

---------------------------------------------------------------------------
Stage 1: measurement-stripping parity
---------------------------------------------------------------------------

In [ ]:
def _measurement_circuit() -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.rz(0.4, 0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

In [ ]:
def test_measurement_removal_parity() -> None:
    qc = _measurement_circuit()
    out_gs = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    out_sk = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=1)
    for out in (out_gs, out_sk):
        assert "measure" not in out.count_ops()
    if is_nwqec_available():
        out_nw = transpile_to_clifford_t_cpp(
            qc.copy(), epsilon=1e-3, forbid_python_fallback=True
        )
        assert "measure" not in out_nw.count_ops()

---------------------------------------------------------------------------
Stage 4: synthesis approximates the original within the requested precision
---------------------------------------------------------------------------

In [ ]:
def _approximation_error(original: QuantumCircuit, approx: QuantumCircuit) -> float:
    """Operator-norm distance up to global phase, on the same qubit count."""
    u_orig = Operator(original).data
    u_approx = Operator(approx).data
    # vdot conjugates its first arg: if u_approx = α·u_orig with |α|=1, then
    # vdot(u_approx, u_orig) = conj(α)·‖u_orig‖², so dividing by |·| gives the
    # phase that maps u_approx onto u_orig. Using vdot(u_orig, u_approx) here
    # would yield α (the inverse direction) and leave a residual of ‖1−α²‖.
    inner = np.vdot(u_approx.flatten(), u_orig.flatten())
    phase = inner / abs(inner) if abs(inner) > 1e-12 else 1.0
    return float(np.linalg.norm(u_orig - phase * u_approx, ord=2))

In [ ]:
def test_gs_final_approximates_input() -> None:
    qc = _rz_circuit_simple()
    out = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    err = _approximation_error(qc, out)
    # Per-Rz precision is 1e-3; circuit has 1 Rz, slack for global behavior.
    assert err < 5e-2, err

In [ ]:
def test_sk_final_approximates_input() -> None:
    qc = _rz_circuit_simple()
    out = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=2)
    err = _approximation_error(qc, out)
    # SK at recursion_degree=2 is loose; just assert it's not catastrophically off.
    assert err < 0.5, err

In [ ]:
def test_nwqec_final_approximates_input() -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    qc = _rz_circuit_simple()
    out = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    err = _approximation_error(qc, out)
    assert err < 5e-2, err

---------------------------------------------------------------------------
Behavioral parity with NWQEC C++: discrete gates must survive untouched
---------------------------------------------------------------------------

In [ ]:
def _broad_basis_circuit() -> QuantumCircuit:
    """Input exercising every discrete gate in the broader basis plus one rz."""
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.t(0)
    qc.tdg(1)
    qc.sdg(0)
    qc.x(0)
    qc.y(1)
    qc.z(0)
    qc.rz(0.3, 0)
    qc.cx(0, 1)
    return qc

In [ ]:
def test_intermediate_preserves_discrete_gates_gs() -> None:
    """Intermediate must contain {t, tdg, sdg, x, y, z} (one each), not decomposed away."""
    qc = _broad_basis_circuit()
    inter, _ = transpile_to_gridsynth_clifford_t(
        qc.copy(), gridsynth_precision=3, return_intermediate=True
    )
    counts = inter.count_ops()
    for g in ("t", "tdg", "sdg", "x", "y", "z"):
        assert counts.get(g, 0) >= 1, f"GS intermediate dropped {g}: {dict(counts)}"

In [ ]:
def test_intermediate_preserves_discrete_gates_sk() -> None:
    qc = _broad_basis_circuit()
    inter, _ = transpile_to_solovay_kitaev_clifford_t(
        qc.copy(), recursion_degree=1, return_intermediate=True
    )
    counts = inter.count_ops()
    for g in ("t", "tdg", "sdg", "x", "y", "z"):
        assert counts.get(g, 0) >= 1, f"SK intermediate dropped {g}: {dict(counts)}"

In [ ]:
def test_intermediate_preserves_discrete_gates_nwqec() -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    qc = _broad_basis_circuit()
    inter, _ = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, return_intermediate=True, forbid_python_fallback=True
    )
    counts = inter.count_ops()
    for g in ("t", "tdg", "sdg", "x", "y", "z"):
        assert counts.get(g, 0) >= 1, f"NWQEC intermediate dropped {g}: {dict(counts)}"

In [ ]:
def test_nwqec_behavioral_parity_final_output() -> None:
    """For the broad-basis input, gates that no synthesis engine emits must be
    preserved exactly (input count = output count). NWQEC C++ already does this
    today; the Python pipelines should match after the basis broadening.

    Note: `t`, `h`, `s`, `tdg`, `sdg` are excluded because synthesis legitimately
    emits them when approximating `rz`. `x` is also excluded from exact parity
    because gridsynth Python's emitted basis includes `X` (it can emit X gates
    nondeterministically depending on the angle); we only assert input ≤ output
    for it.
    """
    qc = _broad_basis_circuit()
    # Input contains one of each; these are NEVER emitted by single-qubit
    # gridsynth/SK output, so output count must equal input count exactly.
    exactly_preserved = {"y": 1, "z": 1, "cx": 1}
    # Gridsynth Python's emitted basis includes X, so the output may have x ≥ input.
    at_least_preserved = {"x": 1}

    out_gs = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    out_sk = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=1)

    for label, out in [("gs", out_gs), ("sk", out_sk)]:
        counts = out.count_ops()
        for gate, want in exactly_preserved.items():
            got = counts.get(gate, 0)
            assert (
                got == want
            ), f"{label}: {gate} count = {got}, want {want}; full {dict(counts)}"
        for gate, lower in at_least_preserved.items():
            got = counts.get(gate, 0)
            assert (
                got >= lower
            ), f"{label}: {gate} count = {got}, want >= {lower}; full {dict(counts)}"

    if is_nwqec_available():
        out_nw = transpile_to_clifford_t_cpp(
            qc.copy(), epsilon=1e-3, forbid_python_fallback=True
        )
        counts = out_nw.count_ops()
        for gate, want in exactly_preserved.items():
            got = counts.get(gate, 0)
            assert (
                got == want
            ), f"nwqec: {gate} count = {got}, want {want}; full {dict(counts)}"
        for gate, lower in at_least_preserved.items():
            got = counts.get(gate, 0)
            assert (
                got >= lower
            ), f"nwqec: {gate} count = {got}, want >= {lower}; full {dict(counts)}"

---------------------------------------------------------------------------
No spurious approximation on already-CT input
---------------------------------------------------------------------------

In [ ]:
def test_single_t_input_not_reapproximated_gs() -> None:
    """qc.t(0) with no rotations should pass through unchanged (no rz->T sequence)."""
    qc = QuantumCircuit(1)
    qc.t(0)
    out = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    assert dict(out.count_ops()) == {"t": 1}

In [ ]:
def test_single_t_input_not_reapproximated_sk() -> None:
    qc = QuantumCircuit(1)
    qc.t(0)
    out = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=1)
    assert dict(out.count_ops()) == {"t": 1}

In [ ]:
def test_single_t_input_not_reapproximated_nwqec() -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    qc = QuantumCircuit(1)
    qc.t(0)
    out = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    assert dict(out.count_ops()) == {"t": 1}

---------------------------------------------------------------------------
End-to-end: PBC converter accepts the final output of every pipeline
(even when it contains x, y, z, sdg in addition to cx, h, s, t, tdg)
---------------------------------------------------------------------------

In [ ]:
def test_pbc_converter_accepts_final_output_gs() -> None:
    from ftcircuitbench.pbc_converter.r_pauli_circ import RotationPauliCirc

    qc = _broad_basis_circuit()
    out = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    rpc = RotationPauliCirc(out)
    assert rpc.process(ifprint=False) is False

In [ ]:
def test_pbc_converter_accepts_final_output_sk() -> None:
    from ftcircuitbench.pbc_converter.r_pauli_circ import RotationPauliCirc

    qc = _broad_basis_circuit()
    out = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=1)
    rpc = RotationPauliCirc(out)
    assert rpc.process(ifprint=False) is False

In [ ]:
def test_pbc_converter_accepts_final_output_nwqec() -> None:
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    from ftcircuitbench.pbc_converter.r_pauli_circ import RotationPauliCirc

    qc = _broad_basis_circuit()
    out = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    rpc = RotationPauliCirc(out)
    assert rpc.process(ifprint=False) is False

---------------------------------------------------------------------------
Reference parity against NWQEC C++ (taken as the ground-truth implementation
of the broader-basis Clifford+T pipeline). Python GS uses the same gridsynth
algorithm as NWQEC, so its gate counts should match exactly. Python SK uses a
different algorithm so we compare unitary distance rather than gate counts.
---------------------------------------------------------------------------

In [ ]:
def _reference_inputs() -> list[QuantumCircuit]:
    """Inputs spanning the broader basis + a pure-rotation circuit."""
    out: list[QuantumCircuit] = [_broad_basis_circuit()]
    qc1 = QuantumCircuit(2)
    qc1.rz(math.pi / 7, 0)
    qc1.cx(0, 1)
    qc1.rz(math.pi / 9, 1)
    out.append(qc1)
    qc2 = QuantumCircuit(1)
    qc2.t(0)
    qc2.rz(0.3, 0)
    qc2.t(0)
    out.append(qc2)
    return out

In [ ]:
@pytest.mark.parametrize("qc", _reference_inputs())
def test_pipelines_preserve_input_discrete_gates(qc: QuantumCircuit) -> None:
    """Behavioral parity: discrete gates present in the input must survive
    transpilation in both Python GS and NWQEC C++. (Synthesis may *add* gates
    like x or t — gridsynth's emitted basis includes them — so the right
    invariant is input ≤ output, not input == output for those.) For
    multi-qubit gates and Paulis that single-qubit synthesis cannot emit,
    the count must be preserved exactly."""
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    out_gs = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    out_nw = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    counts_in = dict(qc.count_ops())
    counts_gs = out_gs.count_ops()
    counts_nw = out_nw.count_ops()

    # Single-qubit synthesis cannot emit cx; Y/Z gates are also outside the
    # gridsynth and SK output bases — these must be preserved exactly.
    exactly_preserved = {"cx", "y", "z"}
    # Discrete gates that synthesis MIGHT emit; the input count is a lower
    # bound on the output count (preservation), not an exact target.
    at_least_preserved = {"x", "tdg", "sdg"}

    for gate in exactly_preserved:
        want = counts_in.get(gate, 0)
        if want > 0:
            assert (
                counts_gs.get(gate, 0) == want
            ), f"GS dropped {gate}: {dict(counts_gs)}"
            assert (
                counts_nw.get(gate, 0) == want
            ), f"NW dropped {gate}: {dict(counts_nw)}"

    for gate in at_least_preserved:
        want = counts_in.get(gate, 0)
        if want > 0:
            assert (
                counts_gs.get(gate, 0) >= want
            ), f"GS dropped {gate}: {dict(counts_gs)}"
            assert (
                counts_nw.get(gate, 0) >= want
            ), f"NW dropped {gate}: {dict(counts_nw)}"

In [ ]:
@pytest.mark.parametrize("qc", _reference_inputs())
def test_gs_python_unitary_matches_nwqec(qc: QuantumCircuit) -> None:
    """Both pipelines call gridsynth at the same precision. The output gate
    sequences may differ slightly (gridsynth makes choices in rounding), but
    the resulting unitaries should be within precision tolerance of each other."""
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    out_gs = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    out_nw = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    fid = process_fidelity(Operator(out_gs), Operator(out_nw))
    # Each pipeline approximates the input within ~1e-3 per rz; combined
    # fidelity should be > 0.999 for circuits with a handful of rzs.
    assert fid > 0.999, f"GS-vs-NWQEC fidelity {fid:.6f} below tolerance"

In [ ]:
@pytest.mark.parametrize("qc", _reference_inputs())
def test_sk_python_unitary_close_to_nwqec(qc: QuantumCircuit) -> None:
    """SK and gridsynth are different algorithms, so gate counts won't match.
    But both approximate the same input unitary — SK at recursion_degree=3
    should be within 0.99 fidelity of the NWQEC output (loose, but a sanity
    check that they're approximating the same thing)."""
    pytest.importorskip("nwqec")
    if not is_nwqec_available():
        pytest.skip("nwqec not available")
    out_sk = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=3)
    out_nw = transpile_to_clifford_t_cpp(
        qc.copy(), epsilon=1e-3, forbid_python_fallback=True
    )
    fid_nw = process_fidelity(Operator(qc), Operator(out_nw))
    fid_sk = process_fidelity(Operator(qc), Operator(out_sk))
    assert fid_nw > 0.999, f"NWQEC reference fidelity {fid_nw:.6f} below tolerance"
    assert (
        fid_sk > 0.5
    ), f"SK fidelity {fid_sk:.6f} too low (recursion_degree=3 was expected to give >= 0.5)"

In [ ]:
@pytest.mark.parametrize("qc", _reference_inputs())
def test_all_three_pipelines_pbc_compatible(qc: QuantumCircuit) -> None:
    """The end-to-end goal: each pipeline's output must feed into PBC successfully."""
    from ftcircuitbench.pbc_converter.r_pauli_circ import RotationPauliCirc

    out_gs = transpile_to_gridsynth_clifford_t(qc.copy(), gridsynth_precision=3)
    out_sk = transpile_to_solovay_kitaev_clifford_t(qc.copy(), recursion_degree=2)

    assert RotationPauliCirc(out_gs).process(ifprint=False) is False
    assert RotationPauliCirc(out_sk).process(ifprint=False) is False

    if is_nwqec_available():
        out_nw = transpile_to_clifford_t_cpp(
            qc.copy(), epsilon=1e-3, forbid_python_fallback=True
        )
        assert RotationPauliCirc(out_nw).process(ifprint=False) is False